In [ ]:
# --- 1. Setup and Library Installation ---
!pip install pandas -q
import pandas as pd
import os
from google.colab import drive
drive.mount('/content/drive')

# --- 2. File Path Configuration and Data Loading ---
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
INPUT_FILE = os.path.join(BASE_DIR, "output_prefname_count.txt")
OUTPUT_MODEL1_FILE = os.path.join(BASE_DIR, "pgp_dataset_for_inhibitor_model_v2.csv") # Specify version
OUTPUT_MODEL2_FILE = os.path.join(BASE_DIR, "pgp_dataset_for_substrate_model_v2.csv") # Specify version

df = pd.read_csv(INPUT_FILE, sep='\t', encoding='shift_jis')
print(f"Successfully loaded {len(df)} compound data entries.")

# --- 3. Reliability Filtering: Total mentions >= 3 ---
df_filtered = df[df['sum_total_count'] >= 3].copy()
print(f"After filtering for >= 3 mentions, {len(df_filtered)} compounds remain.")

# --- 4. Score Calculation ---
epsilon = 1e-9
df_filtered['inhibitor_score'] = df_filtered['inhibitor_count'] / (df_filtered['sum_total_count'] + epsilon)
df_filtered['substrate_score'] = df_filtered['substrate_count'] / (df_filtered['sum_total_count'] + epsilon)
print("Score calculation complete.")

# --- 5. Create Dataset for Model-1 (Inhibitor Model) (✨Applying updated criteria) ---
print("\n--- Creating dataset for Model-1 (Inhibition Prediction Model)... ---")
# Positive: Mentioned as 'inhibitor' >= 3 times & inhibitor_score >= 0.7
inhibitor_positives = df_filtered[
    (df_filtered['inhibitor_count'] >= 3) &
    (df_filtered['inhibitor_score'] >= 0.7)
].copy()
inhibitor_positives['label'] = 1

# Negative: inhibitor_score <= 0.3 (The total mentions >= 3 condition is already applied)
inhibitor_negatives = df_filtered[df_filtered['inhibitor_score'] <= 0.3].copy()
inhibitor_negatives['label'] = 0

df_model1 = pd.concat([inhibitor_positives, inhibitor_negatives], ignore_index=True)
print("Model-1 dataset class distribution:")
print(df_model1['label'].value_counts())
df_model1.to_csv(OUTPUT_MODEL1_FILE, index=False)
print(f"Model-1 dataset saved to: {OUTPUT_MODEL1_FILE}")

# --- 6. Create Dataset for Model-2 (Substrate Model) (✨Applying updated criteria) ---
print("\n--- Creating dataset for Model-2 (Substrate Prediction Model)... ---")
# Positive: Mentioned as 'substrate' >= 3 times & substrate_score >= 0.7
substrate_positives = df_filtered[
    (df_filtered['substrate_count'] >= 3) &
    (df_filtered['substrate_score'] >= 0.7)
].copy()
substrate_positives['label'] = 1

# Negative: substrate_score <= 0.3
substrate_negatives = df_filtered[df_filtered['substrate_score'] <= 0.3].copy()
substrate_negatives['label'] = 0

df_model2 = pd.concat([substrate_positives, substrate_negatives], ignore_index=True)
print("Model-2 dataset class distribution:")
print(df_model2['label'].value_counts())
df_model2.to_csv(OUTPUT_MODEL2_FILE, index=False)
print(f"Model-2 dataset saved to: {OUTPUT_MODEL2_FILE}")

In [ ]:
#이 사이에 baseline으로 RF모델을 사용해보고 AUC를 확인
#RF로 빠르게 baseline AUC를 확인

# --- 1. 라이브러리 임포트 ---
import pandas as pd
import numpy as np
import requests
import time
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import os
from google.colab import drive

# --- 2. 설정 및 데이터 로딩 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
# 모델-1 데이터셋 v2 버전을 사용합니다.
INPUT_CSV_PATH = os.path.join(BASE_DIR, "pgp_dataset_for_inhibitor_model_v2.csv")

print("Random Forest Baseline (모델-1: 저해 예측) 성능 측정을 시작합니다...")
try:
    df = pd.read_csv(INPUT_CSV_PATH)
    print(f"  - {len(df)}개의 화합물 데이터를 로드했습니다.")
except FileNotFoundError:
    print(f"오류: '{INPUT_CSV_PATH}'를 찾을 수 없습니다. 파일 경로를 확인해주세요.")
    exit()

# --- ✨✨✨ 3. SMILES 정보 추가 (오류 해결 파트) ✨✨✨ ---
print("  - PubChem에서 SMILES 정보 가져오는 중...")
smiles_list = []
for index, row in df.iterrows():
    inchikey = row['desalted_inchikey']
    smiles = ''
    if isinstance(inchikey, str) and len(inchikey) > 10:
        try:
            url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchikey}/property/IsomericSMILES/TXT"
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                smiles = response.text.strip()
        except requests.exceptions.RequestException:
            pass
    smiles_list.append(smiles)
    if (index + 1) % 100 == 0:
        print(f"    ... {index + 1} / {len(df)} 처리 완료")
    time.sleep(0.1)

df['smiles'] = smiles_list
print("  - SMILES 정보 추가 완료.")


# --- 4. 특징 공학 (분자 기술자 계산) ---
print("  - 분자 기술자(Descriptor)를 계산하는 중...")
df['mol'] = df['smiles'].apply(lambda x: Chem.MolFromSmiles(x) if pd.notna(x) else None)
df.dropna(subset=['mol'], inplace=True) # SMILES 파싱 실패한 데이터 제거
print(f"  - 유효한 SMILES를 가진 {len(df)}개의 화합물로 진행합니다.")

descriptor_names = [
    'MolWt', 'MolLogP', 'TPSA', 'NumHDonors',
    'NumHAcceptors', 'NumRotatableBonds', 'NumAromaticRings', 'FpDensityMorgan1'
]
def calculate_descriptors(mol):
    return [
        Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol), Descriptors.NumAromaticRings(mol),
        Descriptors.FpDensityMorgan1(mol)
    ]
df[descriptor_names] = df['mol'].apply(lambda mol: pd.Series(calculate_descriptors(mol)))
print("  - 8개의 분자 기술자 계산 완료.")

# --- 5. 모델 훈련 및 5-Fold 교차 검증 ---
X = df[descriptor_names]
y = df['label']
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_aucs = []

print("\n5-Fold 교차 검증을 시작합니다...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    rf_model.fit(X_train, y_train)
    val_probs = rf_model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, val_probs)
    fold_aucs.append(fold_auc)
    print(f"  - Fold {fold} AUC: {fold_auc:.4f}")

# --- 6. 최종 결과 출력 ---
mean_auc = np.mean(fold_aucs)
std_auc = np.std(fold_aucs)
print("\n--- Random Forest Baseline 최종 성능 (모델-1: 저해 예측) ---")
print(f"평균 AUC (5-fold CV): {mean_auc:.4f} ± {std_auc:.4f}")

In [ ]:
# --- 1. 라이브러리 임포트 ---
import pandas as pd
import numpy as np
import requests
import time
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import os
from google.colab import drive

# --- 2. 설정 및 데이터 로딩 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
INPUT_CSV_PATH = os.path.join(BASE_DIR, "pgp_dataset_for_substrate_model_v2.csv")

print("Random Forest Baseline (모델-2: 기질 예측) 성능 측정을 시작합니다...")
try:
    df = pd.read_csv(INPUT_CSV_PATH)
    print(f"  - {len(df)}개의 화합물 데이터를 로드했습니다.")
except FileNotFoundError:
    print(f"오류: '{INPUT_CSV_PATH}'를 찾을 수 없습니다. 파일 경로를 확인해주세요.")
    exit()

# --- 3. SMILES 정보 추가 ---
print("  - PubChem에서 SMILES 정보 가져오는 중...")
smiles_list = []
for index, row in df.iterrows():
    inchikey = row['desalted_inchikey']
    smiles = ''
    if isinstance(inchikey, str) and len(inchikey) > 10:
        try:
            url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchikey}/property/IsomericSMILES/TXT"
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                smiles = response.text.strip()
        except requests.exceptions.RequestException:
            pass
    smiles_list.append(smiles)
    if (index + 1) % 100 == 0:
        print(f"    ... {index + 1} / {len(df)} 처리 완료")
    time.sleep(0.1)

df['smiles'] = smiles_list
print("  - SMILES 정보 추가 완료.")

# --- 4. 특징 공학 (분자 기술자 계산) ---
print("  - 분자 기술자(Descriptor)를 계산하는 중...")
df['mol'] = df['smiles'].apply(lambda x: Chem.MolFromSmiles(x) if pd.notna(x) else None)
df.dropna(subset=['mol'], inplace=True)
print(f"  - 유효한 SMILES를 가진 {len(df)}개의 화합물로 진행합니다.")

descriptor_names = [
    'MolWt', 'MolLogP', 'TPSA', 'NumHDonors',
    'NumHAcceptors', 'NumRotatableBonds', 'NumAromaticRings', 'FpDensityMorgan1'
]
def calculate_descriptors(mol):
    return [
        Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol), Descriptors.NumAromaticRings(mol),
        Descriptors.FpDensityMorgan1(mol)
    ]
df[descriptor_names] = df['mol'].apply(lambda mol: pd.Series(calculate_descriptors(mol)))
print("  - 8개의 분자 기술자 계산 완료.")

# --- 5. 모델 훈련 및 5-Fold 교차 검증 ---
X = df[descriptor_names]
y = df['label']

# ✨✨✨ 'n_jobs=-p1'을 'n_jobs=-1'로 수정한 부분 ✨✨✨
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_aucs = []

print("\n5-Fold 교차 검증을 시작합니다...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    rf_model.fit(X_train, y_train)
    val_probs = rf_model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, val_probs)
    fold_aucs.append(fold_auc)
    print(f"  - Fold {fold} AUC: {fold_auc:.4f}")

# --- 6. 최종 결과 출력 ---
mean_auc = np.mean(fold_aucs)
std_auc = np.std(fold_aucs)
print("\n--- Random Forest Baseline 최종 성능 (모델-2: 기질 예측) ---")
print(f"평균 AUC (5-fold CV): {mean_auc:.4f} ± {std_auc:.4f}")

In [ ]:
# --- 1. 라이브러리 설치 ---
!pip install rdkit-pypi pandas torch torch_geometric requests -q

import pandas as pd
import numpy as np
import requests
import time
import os
import torch
from rdkit import Chem
from rdkit.Chem import Descriptors
from torch_geometric.data import Data
from google.colab import drive

# --- 2. 기본 설정 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
INPUT_MODEL1_CSV = os.path.join(BASE_DIR, "pgp_dataset_for_inhibitor_model_v2.csv")
INPUT_MODEL2_CSV = os.path.join(BASE_DIR, "pgp_dataset_for_substrate_model_v2.csv")
OUTPUT_MODEL1_PT = os.path.join(BASE_DIR, "inhibitor_model_graph_dataset.pt")
OUTPUT_MODEL2_PT = os.path.join(BASE_DIR, "substrate_model_graph_dataset.pt")

# --- 3. 핵심 기능 함수 정의 ---
def add_smiles(df):
    smiles_list = []
    print(f"PubChem에서 SMILES 정보 가져오는 중... (총 {len(df)}개)")
    for index, row in df.iterrows():
        inchikey = row['desalted_inchikey']
        smiles = ''
        if isinstance(inchikey, str) and len(inchikey) > 10:
            try:
                url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchikey}/property/IsomericSMILES/TXT"
                response = requests.get(url, timeout=10)
                if response.status_code == 200:
                    smiles = response.text.strip()
            except requests.exceptions.RequestException:
                pass
        smiles_list.append(smiles)
        if (index + 1) % 100 == 0:
            print(f"  ... {index + 1} / {len(df)} 처리 완료")
        time.sleep(0.1)
    df['smiles'] = smiles_list
    df['smiles'].replace('', np.nan, inplace=True)
    df.dropna(subset=['smiles'], inplace=True)
    print("SMILES 정보 추가 및 결측치 제거 완료.")
    return df

def create_graph_dataset(df, output_path):
    print(f"'{os.path.basename(output_path)}' 생성을 시작합니다...")
    def get_atom_features(atom):
        return [float(f) for f in [atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(), atom.GetNumRadicalElectrons(), atom.GetHybridization(), atom.GetIsAromatic(), atom.IsInRing()]]
    def get_bond_features(bond):
        return [float(f) for f in [bond.GetBondTypeAsDouble(), bond.GetIsAromatic(), bond.IsInRing(), bond.GetIsConjugated()]]

    # ✨✨✨ 바로 이 부분의 'LogP'를 'MolLogP'로 수정했습니다. ✨✨✨
    def calculate_global_descriptors(mol):
        return [Descriptors.MolLogP(mol), Descriptors.MolWt(mol), Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol), Descriptors.NumAromaticRings(mol)]

    def smiles_to_graph(row):
        mol = Chem.MolFromSmiles(row['smiles'])
        if not mol: return None
        atom_features = [get_atom_features(atom) for atom in mol.GetAtoms()]
        edge_indices, edge_features = [], []
        for bond in mol.GetBonds():
            i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            bond_feat = get_bond_features(bond)
            edge_indices.extend([[i, j], [j, i]])
            edge_features.extend([bond_feat, bond_feat])
        global_features = calculate_global_descriptors(mol)
        return Data(x=torch.tensor(atom_features, dtype=torch.float),
                    edge_index=torch.tensor(edge_indices, dtype=torch.long).t().contiguous(),
                    edge_attr=torch.tensor(edge_features, dtype=torch.float),
                    y=torch.tensor([row['label']], dtype=torch.long),
                    global_feat=torch.tensor([global_features], dtype=torch.float))
    graph_list = [graph for _, row in df.iterrows() if (graph := smiles_to_graph(row)) is not None]
    torch.save(graph_list, output_path)
    print(f"성공! 총 {len(graph_list)}개의 그래프를 생성하여 아래 경로에 저장했습니다:")
    print(output_path)

# --- 4. 메인 실행 파트 ---
print(">>> 모델-1 데이터셋 처리 시작 <<<")
df1 = pd.read_csv(INPUT_MODEL1_CSV)
df1_with_smiles = add_smiles(df1)
create_graph_dataset(df1_with_smiles, OUTPUT_MODEL1_PT)

print("\n" + "="*50 + "\n")

print(">>> 모델-2 데이터셋 처리 시작 <<<")
df2 = pd.read_csv(INPUT_MODEL2_CSV)
df2_with_smiles = add_smiles(df2)
create_graph_dataset(df2_with_smiles, OUTPUT_MODEL2_PT)

print("\n모든 작업이 완료되었습니다! 🚀")

In [ ]:
# --- 1. 라이브러리 설치 ---
!pip install optuna -q

import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.nn import GINEConv, global_add_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import optuna
import os
from google.colab import drive

# --- 2. 기본 설정 및 데이터 로딩 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
# 모델-1을 위한 그래프 데이터셋을 불러옵니다.
GRAPH_DATASET_PATH = os.path.join(BASE_DIR, "inhibitor_model_graph_dataset.pt")

print("모델-1 (저해 예측 모델) 훈련 시작...")
try:
    dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"  - {len(dataset)}개의 그래프 데이터를 로드했습니다. ({device} 사용)")
except FileNotFoundError:
    print(f"오류: '{GRAPH_DATASET_PATH}'를 찾을 수 없습니다. 이전 단계를 먼저 실행해주세요.")
    exit()

# 데이터셋의 특징 차원 확인
num_node_features = dataset[0].num_node_features
num_edge_features = dataset[0].num_edge_features
num_global_features = dataset[0].global_feat.shape[1]

# --- 3. GINE 모델 정의 ---
class GINE_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate):
        super(GINE_Model, self).__init__()
        mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINEConv(mlp1, edge_dim=num_edge_features)
        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINEConv(mlp2, edge_dim=num_edge_features)
        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)
    def forward(self, x, edge_index, batch, edge_attr, global_feat):
        x = self.conv1(x, edge_index, edge_attr=edge_attr).relu()
        x = self.conv2(x, edge_index, edge_attr=edge_attr).relu()
        gnn_out = global_add_pool(x, batch)
        combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x)
        x = self.fc2(x)
        return x

# --- 4. Optuna 목적 함수 정의 ---
def objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [64, 128])
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    fold_aucs = []

    # ✨✨✨ 1. 데이터 불균형 해결을 위한 클래스 가중치 계산 ✨✨✨
    num_positives = (labels == 1).sum()
    num_negatives = (labels == 0).sum()
    weight_for_0 = (num_positives + num_negatives) / (2.0 * num_negatives)
    weight_for_1 = (num_positives + num_negatives) / (2.0 * num_positives)
    class_weights = torch.tensor([weight_for_0, weight_for_1], dtype=torch.float).to(device)

    for train_idx, val_idx in skf.split(np.arange(len(dataset)), labels):
        model = GINE_Model(num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        # ✨✨✨ 2. 손실 함수에 계산된 가중치 적용 ✨✨✨
        criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

        # drop_last=True : 마지막 배치의 크기가 1일 경우 발생하는 오류 방지
        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size, drop_last=True)

        for epoch in range(100):
            model.train()
            for data in train_loader:
                data = data.to(device)
                optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()

        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                all_labels.extend(data.y.cpu().tolist())
                all_probs.extend(F.softmax(out, dim=1).cpu()[:, 1].tolist())

        if not all_labels: continue
        fold_aucs.append(roc_auc_score(all_labels, all_probs))

    return np.mean(fold_aucs) if fold_aucs else 0.0

# --- 5. Optuna Study 실행 ---
study = optuna.create_study(direction='maximize', study_name='inhibitor_model_tuning_v2')
study.optimize(objective, n_trials=30)

print("\n--- 모델-1 (저해 예측 모델) Optuna 탐색 완료 ---")
print(f"최고의 평균 AUC (5-fold CV): {study.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study.best_params)

In [ ]:
# --- 1. 라이브러리 설치 ---
!pip install optuna -q

import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.nn import GINEConv, global_add_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import optuna
import os
from google.colab import drive

# --- 2. 기본 설정 및 데이터 로딩 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
# ✨✨✨ 모델-2를 위한 그래프 데이터셋을 불러오도록 경로를 수정했습니다. ✨✨✨
GRAPH_DATASET_PATH = os.path.join(BASE_DIR, "substrate_model_graph_dataset.pt")

print("모델-2 (기질 예측 모델) 훈련 시작...")
try:
    dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"  - {len(dataset)}개의 그래프 데이터를 로드했습니다. ({device} 사용)")
except FileNotFoundError:
    print(f"오류: '{GRAPH_DATASET_PATH}'를 찾을 수 없습니다. 이전 단계를 먼저 실행해주세요.")
    exit()

# 데이터셋의 특징 차원 확인
num_node_features = dataset[0].num_node_features
num_edge_features = dataset[0].num_edge_features
num_global_features = dataset[0].global_feat.shape[1]

# --- 3. GINE 모델 정의 (수정 없음) ---
class GINE_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate):
        super(GINE_Model, self).__init__()
        mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINEConv(mlp1, edge_dim=num_edge_features)
        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINEConv(mlp2, edge_dim=num_edge_features)
        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)
    def forward(self, x, edge_index, batch, edge_attr, global_feat):
        x = self.conv1(x, edge_index, edge_attr=edge_attr).relu()
        x = self.conv2(x, edge_index, edge_attr=edge_attr).relu()
        gnn_out = global_add_pool(x, batch)
        combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x)
        x = self.fc2(x)
        return x

# --- 4. Optuna 목적 함수 정의 (수정 없음) ---
def objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [64, 128])
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    fold_aucs = []
    num_positives = (labels == 1).sum()
    num_negatives = (labels == 0).sum()
    weight_for_0 = (num_positives + num_negatives) / (2.0 * num_negatives)
    weight_for_1 = (num_positives + num_negatives) / (2.0 * num_positives)
    class_weights = torch.tensor([weight_for_0, weight_for_1], dtype=torch.float).to(device)
    for train_idx, val_idx in skf.split(np.arange(len(dataset)), labels):
        model = GINE_Model(num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size, drop_last=True)
        for epoch in range(100):
            model.train()
            for data in train_loader:
                data = data.to(device)
                optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()
        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                all_labels.extend(data.y.cpu().tolist())
                all_probs.extend(F.softmax(out, dim=1).cpu()[:, 1].tolist())
        if not all_labels: continue
        fold_aucs.append(roc_auc_score(all_labels, all_probs))
    return np.mean(fold_aucs) if fold_aucs else 0.0

# --- 5. Optuna Study 실행 (Study 이름 수정) ---
study_model2 = optuna.create_study(direction='maximize', study_name='substrate_model_tuning_v2')
study_model2.optimize(objective, n_trials=30)

print("\n--- 모델-2 (기질 예측 모델) Optuna 탐색 완료 ---")
print(f"최고의 평균 AUC (5-fold CV): {study_model2.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study_model2.best_params)

In [ ]:
# --- 1. 라이브러리 임포트 ---
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.nn import GINEConv, global_add_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from google.colab import drive

# --- 2. 기본 설정 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 데이터셋 경로
INHIBITOR_DATASET_PATH = os.path.join(BASE_DIR, "inhibitor_model_graph_dataset.pt")
SUBSTRATE_DATASET_PATH = os.path.join(BASE_DIR, "substrate_model_graph_dataset.pt")

# Optuna 최적화 결과
params_inhibitor = {'lr': 0.00010030692618719764, 'hidden_channels': 128, 'batch_size': 64, 'dropout_rate': 0.48791402745092816}
params_substrate = {'lr': 0.000408429094426728, 'hidden_channels': 128, 'batch_size': 16, 'dropout_rate': 0.25145428588551944}

# --- 3. GINE 모델 정의 ---
class GINE_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate):
        super(GINE_Model, self).__init__()
        mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINEConv(mlp1, edge_dim=num_edge_features)
        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINEConv(mlp2, edge_dim=num_edge_features)
        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)
    def forward(self, x, edge_index, batch, edge_attr, global_feat):
        x = self.conv1(x, edge_index, edge_attr=edge_attr).relu(); x = self.conv2(x, edge_index, edge_attr=edge_attr).relu()
        gnn_out = global_add_pool(x, batch); combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x); x = self.fc2(x)
        return x

# --- 4. Out-of-Fold 예측 확률 계산 함수 ---
def get_oof_predictions(dataset, params):
    print(f"\n'{dataset[0].name if hasattr(dataset[0], 'name') else 'Unknown'}' 데이터셋에 대한 Out-of-Fold 예측을 시작합니다...")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])

    oof_preds = np.zeros(len(dataset))
    # 데이터 객체에 화합물 이름이 없을 경우를 대비하여 인덱스를 이름으로 사용
    oof_names = [d.name if hasattr(d, 'name') else f"compound_{i}" for i, d in enumerate(dataset)]

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(len(dataset)), labels), 1):
        print(f"  - Fold {fold}/5 훈련 및 예측 중...")

        train_subset = [dataset[i] for i in train_idx]
        val_subset = [dataset[i] for i in val_idx]

        model = GINE_Model(dataset[0].num_node_features, dataset[0].num_edge_features, dataset[0].global_feat.shape[1],
                         params['hidden_channels'], params['dropout_rate']).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'])
        criterion = torch.nn.CrossEntropyLoss()
        train_loader = DataLoader(train_subset, batch_size=params['batch_size'], shuffle=True)

        for epoch in range(100):
            model.train()
            for data in train_loader:
                data = data.to(DEVICE); optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                loss = criterion(out, data.y); loss.backward(); optimizer.step()

        model.eval()
        val_loader = DataLoader(val_subset, batch_size=params['batch_size'])
        with torch.no_grad():
            # 예측 확률을 저장할 임시 리스트
            temp_probs = []
            for data in val_loader:
                data = data.to(DEVICE)
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                probs = F.softmax(out, dim=1)[:, 1].cpu().numpy()
                temp_probs.extend(probs)
            # 원본 인덱스에 예측 확률을 저장
            oof_preds[val_idx] = temp_probs

    print("OOF 예측 완료!")
    return pd.DataFrame({'name': oof_names, 'prob': oof_preds})

# --- 5. 메인 실행 ---
inhibitor_dataset = torch.load(INHIBITOR_DATASET_PATH, weights_only=False)
inhibitor_preds_df = get_oof_predictions(inhibitor_dataset, params_inhibitor).rename(columns={'prob': 'P_inhibition'})

substrate_dataset = torch.load(SUBSTRATE_DATASET_PATH, weights_only=False)
substrate_preds_df = get_oof_predictions(substrate_dataset, params_substrate).rename(columns={'prob': 'P_substrate'})

final_results = pd.merge(inhibitor_preds_df, substrate_preds_df, on='name', how='outer')

# --- 6. 최종 결과 시각화 ---
print("\n화합물 분포 지도를 생성합니다...")
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(12, 12))
p = sns.scatterplot(data=final_results, x='P_inhibition', y='P_substrate', alpha=0.7)
p.set_xlabel("P(inhibition) - 저해 확률 (OOF)", fontsize=14)
p.set_ylabel("P(substrate) - 기질 확률 (OOF)", fontsize=14)
p.set_title("P-gp Interaction Profile Map (Out-of-Fold Predictions)", fontsize=18, pad=20)
plt.axhline(0.5, color='grey', linestyle='--'); plt.axvline(0.5, color='grey', linestyle='--')
plt.text(0.95, 0.05, "Specific Inhibitors", ha='right', va='bottom', fontsize=12, color='red', alpha=0.8)
plt.text(0.05, 0.95, "Specific Substrates", ha='left', va='top', fontsize=12, color='blue', alpha=0.8)
plt.text(0.95, 0.95, "Dual-Role", ha='right', va='top', fontsize=12, color='purple', alpha=0.8)
plt.text(0.05, 0.05, "Non-Interactors", ha='left', va='bottom', fontsize=12, color='green', alpha=0.8)
plt.show()

print("\n--- 최종 예측 확률 샘플 (name 컬럼으로 merge된 결과) ---")
print(final_results.head(10))

In [ ]:
# --- 1. 라이브러리 임포트 ---
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import optuna
from torch_geometric.nn import GINEConv, global_add_pool
import matplotlib.pyplot as plt
import os
from google.colab import drive

# --- 2. 기본 설정 및 데이터 로딩 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ✨✨✨ Optuna 실행 전, 데이터셋을 먼저 불러옵니다. ✨✨✨
GRAPH_DATASET_PATH = os.path.join(BASE_DIR, "inhibitor_model_graph_dataset.pt")
print("모델-1 (저해 예측 모델) 데이터셋 로딩 중...")
try:
    dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)
    print(f"  - 성공! {len(dataset)}개의 그래프 데이터를 로드했습니다.")
except FileNotFoundError:
    print(f"오류: '{GRAPH_DATASET_PATH}'를 찾을 수 없습니다. 데이터 생성 코드를 먼저 실행해주세요.")
    # 데이터셋이 없으면 여기서 중단
    exit()

# --- 3. 전역 변수 및 모델 정의 ---
num_node_features = dataset[0].num_node_features
num_edge_features = dataset[0].num_edge_features
num_global_features = dataset[0].global_feat.shape[1]

class GINE_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate):
        super(GINE_Model, self).__init__()
        mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINEConv(mlp1, edge_dim=num_edge_features)
        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINEConv(mlp2, edge_dim=num_edge_features)
        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)

    def forward(self, x, edge_index, batch, edge_attr, global_feat):
        x = self.conv1(x, edge_index, edge_attr=edge_attr).relu()
        x = self.conv2(x, edge_index, edge_attr=edge_attr).relu()
        gnn_out = global_add_pool(x, batch)
        combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x)
        x = self.fc2(x)
        return x

# --- 4. Optuna 목적 함수 ---
def objective(trial, dataset):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [32, 64, 128])
    batch_size = trial.suggest_categorical('batch_size', [16, 32])
    dropout_rate = trial.suggest_float('dropout_rate', 0.25, 0.6)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    fold_val_losses, last_fold_history = [], {}

    # 클래스 가중치 계산
    num_positives = (labels == 1).sum()
    num_negatives = (labels == 0).sum()
    weight_for_0 = (num_positives + num_negatives) / (2.0 * num_negatives) if num_negatives > 0 else 1.0
    weight_for_1 = (num_positives + num_negatives) / (2.0 * num_positives) if num_positives > 0 else 1.0
    class_weights = torch.tensor([weight_for_0, weight_for_1], dtype=torch.float).to(DEVICE)

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(len(dataset)), labels)):
        model = GINE_Model(num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        # 가중 손실 함수 적용
        criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size, drop_last=True)
        train_losses, val_losses, val_aucs = [], [], []

        for epoch in range(75):
            model.train()
            epoch_train_loss = 0
            for data in train_loader:
                data = data.to(DEVICE); optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                loss = criterion(out, data.y); loss.backward(); optimizer.step()
                epoch_train_loss += loss.item()
            if len(train_loader) > 0: train_losses.append(epoch_train_loss / len(train_loader))

            model.eval()
            epoch_val_loss = 0
            all_labels, all_probs = [], []
            with torch.no_grad():
                for data in val_loader:
                    data = data.to(DEVICE)
                    out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                    loss = criterion(out, data.y)
                    epoch_val_loss += loss.item()
                    all_labels.extend(data.y.cpu().tolist())
                    all_probs.extend(F.softmax(out, dim=1).cpu()[:, 1].tolist())

            if not all_labels: continue
            if len(val_loader) > 0: val_losses.append(epoch_val_loss / len(val_loader))
            val_aucs.append(roc_auc_score(all_labels, all_probs))

        if val_losses: fold_val_losses.append(val_losses[-1])
        if fold == skf.get_n_splits() - 1:
            last_fold_history = {'train_loss': train_losses, 'val_loss': val_losses, 'val_auc': val_aucs}

    trial.set_user_attr("history", last_fold_history)
    if not fold_val_losses: return float('inf')
    return np.mean(fold_val_losses)

# --- 5. Optuna Study 실행 ---
print("\n--- Optuna 최적화 실행 (목표: Validation Loss 최소화) ---")
study = optuna.create_study(direction='minimize', study_name='gine_loss_minimization_inhibitor')
study.optimize(lambda trial: objective(trial, dataset), n_trials=30)

print("\n--- [Loss 최소화] 모델-1 Optuna 탐색 완료 ---")
print(f"최저 평균 Validation Loss (5-fold CV): {study.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study.best_params)

# --- 6. 학습 곡선 시각화 ---
print("\n--- 최고 성능 Trial의 학습 곡선 ---")
best_history = study.best_trial.user_attrs.get("history", {})
if best_history and best_history.get('train_loss'):
    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)
    plt.plot(best_history['train_loss'], label='Train Loss', color='blue')
    plt.plot(best_history['val_loss'], label='Validation Loss', color='orange')
    plt.title('Loss Curve for Best Trial', fontsize=16)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend(); plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(best_history['val_auc'], label='Validation AUC', color='green')
    best_auc_epoch = np.argmax(best_history['val_auc'])
    best_auc_value = np.max(best_history['val_auc'])
    plt.scatter(best_auc_epoch, best_auc_value, color='red', zorder=5, label=f'Best AUC: {best_auc_value:.4f} at Epoch {best_auc_epoch+1}')
    plt.title('AUC Curve for Best Trial', fontsize=16)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('AUC', fontsize=12)
    plt.legend(); plt.grid(True)

    plt.tight_layout(); plt.show()
else:
    print("학습 기록을 찾을 수 없습니다.")

In [ ]:
# --- 1. 라이브러리 임포트 ---
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import optuna
from torch_geometric.nn import GINEConv, global_add_pool
import matplotlib.pyplot as plt
import os
from google.colab import drive

# --- 2. 기본 설정 및 데이터 로딩 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ✨✨✨ 모델-2를 위한 그래프 데이터셋을 불러옵니다. ✨✨✨
GRAPH_DATASET_PATH = os.path.join(BASE_DIR, "substrate_model_graph_dataset.pt")
print("모델-2 (기질 예측 모델) 데이터셋 로딩 중...")
try:
    dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)
    print(f"  - 성공! {len(dataset)}개의 그래프 데이터를 로드했습니다.")
except FileNotFoundError:
    print(f"오류: '{GRAPH_DATASET_PATH}'를 찾을 수 없습니다. 데이터 생성 코드를 먼저 실행해주세요.")
    exit()

# --- 3. 전역 변수 및 모델 정의 ---
num_node_features = dataset[0].num_node_features
num_edge_features = dataset[0].num_edge_features
num_global_features = dataset[0].global_feat.shape[1]

class GINE_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate):
        super(GINE_Model, self).__init__()
        mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINEConv(mlp1, edge_dim=num_edge_features)
        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINEConv(mlp2, edge_dim=num_edge_features)
        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)
    def forward(self, x, edge_index, batch, edge_attr, global_feat):
        x = self.conv1(x, edge_index, edge_attr=edge_attr).relu(); x = self.conv2(x, edge_index, edge_attr=edge_attr).relu()
        gnn_out = global_add_pool(x, batch); combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x); x = self.fc2(x)
        return x

# --- 4. Optuna 목적 함수 ---
def objective(trial, dataset):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [32, 64, 128])
    batch_size = trial.suggest_categorical('batch_size', [16, 32])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.6)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    fold_val_losses, last_fold_history = [], {}
    num_positives = (labels == 1).sum(); num_negatives = (labels == 0).sum()
    weight_for_0 = (num_positives + num_negatives) / (2.0 * num_negatives) if num_negatives > 0 else 1.0
    weight_for_1 = (num_positives + num_negatives) / (2.0 * num_positives) if num_positives > 0 else 1.0
    class_weights = torch.tensor([weight_for_0, weight_for_1], dtype=torch.float).to(DEVICE)
    for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(len(dataset)), labels)):
        model = GINE_Model(num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size, drop_last=True)
        train_losses, val_losses, val_aucs = [], [], []
        for epoch in range(75):
            model.train()
            epoch_train_loss = 0
            for data in train_loader:
                data = data.to(DEVICE); optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                loss = criterion(out, data.y); loss.backward(); optimizer.step()
                epoch_train_loss += loss.item()
            if len(train_loader) > 0: train_losses.append(epoch_train_loss / len(train_loader))
            model.eval()
            epoch_val_loss = 0
            all_labels, all_probs = [], []
            with torch.no_grad():
                for data in val_loader:
                    data = data.to(DEVICE)
                    out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                    loss = criterion(out, data.y)
                    epoch_val_loss += loss.item()
                    all_labels.extend(data.y.cpu().tolist())
                    all_probs.extend(F.softmax(out, dim=1).cpu()[:, 1].tolist())
            if not all_labels: continue
            if len(val_loader) > 0: val_losses.append(epoch_val_loss / len(val_loader))
            val_aucs.append(roc_auc_score(all_labels, all_probs))
        if val_losses: fold_val_losses.append(val_losses[-1])
        if fold == skf.get_n_splits() - 1:
            last_fold_history = {'train_loss': train_losses, 'val_loss': val_losses, 'val_auc': val_aucs}
    trial.set_user_attr("history", last_fold_history)
    if not fold_val_losses: return float('inf')
    return np.mean(fold_val_losses)

# --- 5. Optuna Study 실행 ---
print("\n--- Optuna 최적화 실행 (목표: Validation Loss 최소화) ---")
study = optuna.create_study(direction='minimize', study_name='gine_loss_minimization_substrate')
study.optimize(lambda trial: objective(trial, dataset), n_trials=30)

print("\n--- [Loss 최소화] 모델-2 Optuna 탐색 완료 ---")
print(f"최저 평균 Validation Loss (5-fold CV): {study.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study.best_params)

# --- 6. 학습 곡선 시각화 ---
print("\n--- 최고 성능 Trial의 학습 곡선 ---")
best_history = study.best_trial.user_attrs.get("history", {})
if best_history and best_history.get('train_loss'):
    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1); plt.plot(best_history['train_loss'], label='Train Loss', color='blue'); plt.plot(best_history['val_loss'], label='Validation Loss', color='orange')
    plt.title('Loss Curve for Best Trial'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)
    plt.subplot(1, 2, 2); plt.plot(best_history['val_auc'], label='Validation AUC', color='green')
    best_auc_epoch = np.argmax(best_history['val_auc']); best_auc_value = np.max(best_history['val_auc'])
    plt.scatter(best_auc_epoch, best_auc_value, color='red', zorder=5, label=f'Best AUC: {best_auc_value:.4f} at Epoch {best_auc_epoch+1}')
    plt.title('AUC Curve for Best Trial'); plt.xlabel('Epoch'); plt.ylabel('AUC'); plt.legend(); plt.grid(True)
    plt.tight_layout(); plt.show()
else:
    print("학습 기록을 찾을 수 없습니다.")

In [ ]:
# --- 1. 라이브러리 임포트 ---
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.nn import GINEConv, global_add_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from google.colab import drive

# --- 2. 기본 설정 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 데이터셋 경로
INHIBITOR_DATASET_PATH = os.path.join(BASE_DIR, "inhibitor_model_graph_dataset.pt")
SUBSTRATE_DATASET_PATH = os.path.join(BASE_DIR, "substrate_model_graph_dataset.pt")
INHIBITOR_CSV_PATH = os.path.join(BASE_DIR, "pgp_dataset_for_inhibitor_model_v2.csv")
SUBSTRATE_CSV_PATH = os.path.join(BASE_DIR, "pgp_dataset_for_substrate_model_v2.csv")


# Optuna 최적화 결과
params_inhibitor = {'lr': 0.0001976113883653995, 'hidden_channels': 64, 'batch_size': 32, 'dropout_rate': 0.45631011478825545}
params_substrate = {'lr': 0.00024644109016033717, 'hidden_channels': 64, 'batch_size': 16, 'dropout_rate': 0.5269097585930824}

# --- 3. GINE 모델 정의 ---
class GINE_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate):
        super(GINE_Model, self).__init__()
        mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINEConv(mlp1, edge_dim=num_edge_features)
        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINEConv(mlp2, edge_dim=num_edge_features)
        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)
    def forward(self, x, edge_index, batch, edge_attr, global_feat):
        x = self.conv1(x, edge_index, edge_attr=edge_attr).relu(); x = self.conv2(x, edge_index, edge_attr=edge_attr).relu()
        gnn_out = global_add_pool(x, batch); combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x); x = self.fc2(x)
        return x

# --- 4. Out-of-Fold 예측 확률 계산 함수 ---
def get_oof_predictions(dataset, csv_path, params, model_name):
    print(f"\n{model_name} 데이터셋에 대한 Out-of-Fold 예측을 시작합니다...")

    # CSV에서 화합물 이름(name)을 미리 로드
    df_names = pd.read_csv(csv_path)
    # 그래프 데이터셋 생성 시 SMILES 파싱 실패 등으로 일부가 빠졌을 수 있으므로, 실제 dataset 길이에 맞춰 이름을 슬라이싱
    names_list = df_names['desalted_pref_name'].tolist()[:len(dataset)]

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])

    oof_preds = np.zeros(len(dataset))

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(len(dataset)), labels), 1):
        print(f"  - Fold {fold}/5 훈련 및 예측 중...")

        train_subset = [dataset[i] for i in train_idx]
        val_subset = [dataset[i] for i in val_idx]

        model = GINE_Model(dataset[0].num_node_features, dataset[0].num_edge_features, dataset[0].global_feat.shape[1],
                         params['hidden_channels'], params['dropout_rate']).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'])
        criterion = torch.nn.CrossEntropyLoss() # 훈련 시에는 가중치 없이 진행
        train_loader = DataLoader(train_subset, batch_size=params['batch_size'], shuffle=True, drop_last=True)

        for epoch in range(100):
            model.train()
            for data in train_loader:
                data = data.to(DEVICE); optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                loss = criterion(out, data.y); loss.backward(); optimizer.step()

        model.eval()
        val_loader = DataLoader(val_subset, batch_size=params['batch_size'])
        with torch.no_grad():
            temp_probs = []
            for data in val_loader:
                data = data.to(DEVICE)
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                probs = F.softmax(out, dim=1)[:, 1].cpu().numpy()
                temp_probs.extend(probs)
            oof_preds[val_idx] = temp_probs

    print("OOF 예측 완료!")
    return pd.DataFrame({'name': names_list, 'prob': oof_preds})

# --- 5. 메인 실행 ---
inhibitor_dataset = torch.load(INHIBITOR_DATASET_PATH, weights_only=False)
inhibitor_preds_df = get_oof_predictions(inhibitor_dataset, INHIBITOR_CSV_PATH, params_inhibitor, "Model-1").rename(columns={'prob': 'P_inhibition'})

substrate_dataset = torch.load(SUBSTRATE_DATASET_PATH, weights_only=False)
substrate_preds_df = get_oof_predictions(substrate_dataset, SUBSTRATE_CSV_PATH, params_substrate, "Model-2").rename(columns={'prob': 'P_substrate'})

# 화합물 이름(name)을 기준으로 두 예측 결과를 병합합니다. 양쪽에 모두 있는 화합물만 남깁니다.
final_results = pd.merge(inhibitor_preds_df, substrate_preds_df, on='name', how='inner')

# --- 6. 최종 결과 시각화 ---
print("\n화합물 분포 지도를 생성합니다...")
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(12, 12))
p = sns.scatterplot(data=final_results, x='P_inhibition', y='P_substrate', alpha=0.7, s=50)
p.set_xlabel("P(inhibition) - 저해 확률 (OOF)", fontsize=14)
p.set_ylabel("P(substrate) - 기질 확률 (OOF)", fontsize=14)
p.set_title("P-gp Interaction Profile Map (Out-of-Fold Predictions)", fontsize=18, pad=20)
plt.axhline(0.5, color='grey', linestyle='--'); plt.axvline(0.5, color='grey', linestyle='--')
plt.text(0.95, 0.05, "Specific Inhibitors", ha='right', va='bottom', fontsize=12, color='red', alpha=0.8)
plt.text(0.05, 0.95, "Specific Substrates", ha='left', va='top', fontsize=12, color='blue', alpha=0.8)
plt.text(0.95, 0.95, "Dual-Role", ha='right', va='top', fontsize=12, color='purple', alpha=0.8)
plt.text(0.05, 0.05, "Non-Interactors", ha='left', va='bottom', fontsize=12, color='green', alpha=0.8)
plt.xlim(-0.05, 1.05); plt.ylim(-0.05, 1.05)
plt.show()

print("\n--- 최종 예측 확률 샘플 ---")
print(final_results.head(10))

In [ ]:
# --- 0. 라이브러리 설치 (가장 먼저 실행!) ---
# This command installs the necessary libraries for graph deep learning and cheminformatics.

!pip install torch_geometric rdkit-pypi -q

# --- 1. 라이브러리 임포트 ---
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.nn import GINEConv, global_add_pool
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
from torch_geometric.explain import Explainer, GNNExplainer
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display, Image
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
import os
import time
import requests
from google.colab import drive

# --- 2. 기본 설정 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INHIBITOR_DATASET_PATH = os.path.join(BASE_DIR, "inhibitor_model_graph_dataset.pt")
SUBSTRATE_DATASET_PATH = os.path.join(BASE_DIR, "substrate_model_graph_dataset.pt")
INHIBITOR_CSV_PATH = os.path.join(BASE_DIR, "pgp_dataset_for_inhibitor_model_v2.csv")
SUBSTRATE_CSV_PATH = os.path.join(BASE_DIR, "pgp_dataset_for_substrate_model_v2.csv")
params_inhibitor = {'lr': 0.0001976113883653995, 'hidden_channels': 64, 'batch_size': 32, 'dropout_rate': 0.45631011478825545}
params_substrate = {'lr': 0.00024644109016033717, 'hidden_channels': 64, 'batch_size': 16, 'dropout_rate': 0.5269097585930824}

# --- 3. GINE 모델 정의 ---
class GINE_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate):
        super(GINE_Model, self).__init__()
        mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINEConv(mlp1, edge_dim=num_edge_features)
        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINEConv(mlp2, edge_dim=num_edge_features)
        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)
    def forward(self, x, edge_index, batch, edge_attr, global_feat):
        x = self.conv1(x, edge_index, edge_attr=edge_attr).relu(); x = self.conv2(x, edge_index, edge_attr=edge_attr).relu()
        gnn_out = global_add_pool(x, batch); combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x); x = self.fc2(x)
        return x

# --- 4. OOF 예측 및 그룹 선별 ---
def get_oof_predictions(dataset, csv_path, params, model_name):
    print(f"\nRunning Out-of-Fold predictions for {model_name}...")
    df_names = pd.read_csv(csv_path)
    # Ensure names_list length matches dataset length in case some molecules failed to parse
    names_list = df_names['desalted_pref_name'].tolist()[:len(dataset)]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    oof_preds = np.zeros(len(dataset))

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(len(dataset)), labels), 1):
        train_subset = [dataset[i] for i in train_idx]; val_subset = [dataset[i] for i in val_idx]
        model = GINE_Model(dataset[0].num_node_features, dataset[0].num_edge_features, dataset[0].global_feat.shape[1],
                         params['hidden_channels'], params['dropout_rate']).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'])
        criterion = torch.nn.CrossEntropyLoss()
        train_loader = DataLoader(train_subset, batch_size=params['batch_size'], shuffle=True, drop_last=True)
        for epoch in range(100):
            model.train()
            for data in train_loader:
                data = data.to(DEVICE); optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                loss = criterion(out, data.y); loss.backward(); optimizer.step()

        model.eval()
        # Use a temporary list to store predictions for the current fold
        temp_probs = []
        # Create val_loader inside the fold loop, ensuring correct batching
        val_loader = DataLoader(val_subset, batch_size=params['batch_size'])
        with torch.no_grad():
            for data in val_loader:
                data = data.to(DEVICE)
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                probs = F.softmax(out, dim=1)[:, 1].cpu().numpy()
                temp_probs.extend(probs)
        # Assign predictions back to the correct indices in the main array
        oof_preds[val_idx] = temp_probs

    print(f"OOF prediction for {model_name} complete!")
    return pd.DataFrame({'name': names_list, 'prob': oof_preds})

# Run predictions for both models
inhibitor_dataset = torch.load(INHIBITOR_DATASET_PATH, weights_only=False)
inhibitor_preds_df = get_oof_predictions(inhibitor_dataset, INHIBITOR_CSV_PATH, params_inhibitor, "Model-1 (Inhibitor)").rename(columns={'prob': 'P_inhibition'})

substrate_dataset = torch.load(SUBSTRATE_DATASET_PATH, weights_only=False)
substrate_preds_df = get_oof_predictions(substrate_dataset, SUBSTRATE_CSV_PATH, params_substrate, "Model-2 (Substrate)").rename(columns={'prob': 'P_substrate'})

final_results = pd.merge(inhibitor_preds_df, substrate_preds_df, on='name', how='inner')

# ✨✨✨ Threshold를 0.7과 0.3으로 수정한 부분 ✨✨✨
specific_inhibitors_df = final_results[
    (final_results['P_inhibition'] >= 0.7) &  # Adjusted from 0.8
    (final_results['P_substrate'] <= 0.3)   # Adjusted from 0.2
]
TARGET_COMPOUNDS = specific_inhibitors_df['name'].tolist()
print(f"\nTotal of {len(TARGET_COMPOUNDS)} 'Specific Inhibitors' selected with 0.7/0.3 criteria.")
print(TARGET_COMPOUNDS)
# --- 5. 최종 저해제 모델 훈련 및 데이터 준비 ---
print("\nSHAP 분석을 위한 최종 저해제 모델을 훈련합니다...")
df_inhibitor = pd.read_csv(INHIBITOR_CSV_PATH)

# ✨✨✨ 핵심 수정 부분 시작 ✨✨✨
# GNNExplainer 분석에 필요한 SMILES 정보를 PubChem에서 다시 가져옵니다.
# (이전에 만든 inhibitor_dataset에는 SMILES와 이름 정보가 없기 때문입니다.)
print("  - 분석에 필요한 SMILES 정보를 다시 가져옵니다...")
smiles_list = []
for index, row in df_inhibitor.iterrows():
    inchikey = row['desalted_inchikey']
    smiles = ''
    if isinstance(inchikey, str) and len(inchikey) > 10:
        try:
            url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchikey}/property/IsomericSMILES/TXT"
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                smiles = response.text.strip()
        except requests.exceptions.RequestException:
            pass
    smiles_list.append(smiles)
    time.sleep(0.05)

df_inhibitor['smiles'] = smiles_list
df_inhibitor.dropna(subset=['smiles'], inplace=True)
df_inhibitor = df_inhibitor[df_inhibitor['smiles'] != ''].copy()

# inhibitor_dataset의 각 그래프 객체에 이름과 SMILES 정보를 "이름표"처럼 붙여줍니다.
# 이 작업은 데이터셋을 생성한 순서와 CSV 파일의 순서가 동일하다고 가정합니다.
for i, data in enumerate(inhibitor_dataset):
    if i < len(df_inhibitor):
        data.name = df_inhibitor.iloc[i]['desalted_pref_name']
        data.smiles = df_inhibitor.iloc[i]['smiles']

print("  - 그래프 데이터에 이름과 SMILES 정보 추가 완료.")
# ✨✨✨ 핵심 수정 부분 끝 ✨✨✨

# 데이터셋의 특징 차원 확인
num_node_features = inhibitor_dataset[0].num_node_features
num_edge_features = inhibitor_dataset[0].num_edge_features
num_global_features = inhibitor_dataset[0].global_feat.shape[1]

# 최종 모델 생성
final_model = GINE_Model(num_node_features, num_edge_features, num_global_features,
                       params_inhibitor['hidden_channels'], params_inhibitor['dropout_rate']).to(DEVICE)
optimizer = torch.optim.Adam(final_model.parameters(), lr=params_inhibitor['lr'])
criterion = torch.nn.CrossEntropyLoss()
train_loader = DataLoader(inhibitor_dataset, batch_size=params_inhibitor['batch_size'], shuffle=True, drop_last=True)

# 최종 모델 훈련
print("최종 모델 훈련 시작...")
for epoch in range(100):
    final_model.train()
    for data in train_loader:
        data = data.to(DEVICE); optimizer.zero_grad()
        out = final_model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
        loss = criterion(out, data.y); loss.backward(); optimizer.step()
print("최종 모델 훈련 완료!")


# --- 6. GNNExplainer 분석 실행 ---
explainer = Explainer(
    model=final_model, algorithm=GNNExplainer(epochs=200),
    explanation_type='model', node_mask_type='attributes', edge_mask_type='object',
    model_config=dict(mode='binary_classification', task_level='graph', return_type='raw'),
)

print("\n--- GNNExplainer 분석 시작 ---")
for compound_name in TARGET_COMPOUNDS[:5]: # 상위 5개 화합물 분석
    try:
        # 이제 각 그래프에 .name 속성이 있으므로 정상적으로 찾을 수 있습니다.
        target_graph = next(g for g in inhibitor_dataset if hasattr(g, 'name') and g.name.lower() == compound_name.lower())

        # 해당 그래프에 .smiles 속성이 있는지 다시 한번 확인
        if not hasattr(target_graph, 'smiles') or not target_graph.smiles:
            print(f"\nWarning: SMILES string not found for '{compound_name}'. Skipping.")
            continue

        explanation = explainer(
            x=target_graph.x.to(DEVICE), edge_index=target_graph.edge_index.to(DEVICE),
            batch=torch.zeros(target_graph.x.size(0), dtype=torch.long).to(DEVICE),
            edge_attr=target_graph.edge_attr.to(DEVICE), global_feat=target_graph.global_feat.to(DEVICE))

        mol = Chem.MolFromSmiles(target_graph.smiles)
        node_mask = explanation.node_mask.cpu().numpy().flatten()

        # 원자별 중요도를 0과 1 사이로 정규화하여 색상에 반영
        norm_mask = (node_mask - node_mask.min()) / (node_mask.max() - node_mask.min() + 1e-9)
        atom_colors = {i: (1.0, 1.0 - v, 0.0) for i, v in enumerate(norm_mask)}

        img = Draw.MolToImage(
            mol,
            highlightAtoms=list(range(mol.GetNumAtoms())),
            highlightAtomColors=atom_colors,
            size=(350, 300)
        )

        print(f"\n--- Explanation for: {compound_name} ---")
        display(img)

    except StopIteration:
        print(f"\nWarning: '{compound_name}' was not found in the inhibitor dataset for explanation.")
    except Exception as e:
        print(f"\nAn error occurred for '{compound_name}': {e}")

In [ ]:
# --- 1단계: 환경 설정 (이 셀만 먼저 실행해주세요) ---

# 1. NumPy 버전을 RDKit과 호환되는 1.x대로 낮춥니다.
!pip uninstall numpy -y
!pip install "numpy<2.0"

# 2. 나머지 필수 라이브러리들을 설치합니다.
!pip install chembl_webresource_client pandas rdkit-pypi torch torch_geometric scikit-learn -q

print("\n✅ 환경 설정이 완료되었습니다. 다음 단계로 진행해주세요.")

In [ ]:
# --- 2단계: 외부 검증 실행 (런타임 다시 시작 후 이 셀을 실행하세요) ---

# --- 라이브러리 임포트 ---
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import Descriptors
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.nn import GINEConv, global_add_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import roc_auc_score
import os
from google.colab import drive

# --- 기본 설정 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# --- 우리 모델의 '지인 목록' (Known InChIKeys) 만들기 ---
print("\n--- Step 1: Creating a list of known compounds... ---")
try:
    df_inhib_model = pd.read_csv(os.path.join(BASE_DIR, "pgp_dataset_for_inhibitor_model_v2.csv"))
    df_subs_model = pd.read_csv(os.path.join(BASE_DIR, "pgp_dataset_for_substrate_model_v2.csv"))
    known_inchikeys = set(df_inhib_model['desalted_inchikey'].dropna())
    known_inchikeys.update(set(df_subs_model['desalted_inchikey'].dropna()))
    print(f"  - Total of {len(known_inchikeys)} unique InChIKeys loaded from our datasets.")
except FileNotFoundError as e:
    print(f"Error: Could not find dataset files. Please check the file paths. {e}")
    exit()

# --- ChEMBL에서 '외부 손님' (IC50 데이터) 가져오기 ---
print("\n--- Step 2: Fetching external validation data from ChEMBL... ---")
activities = new_client.activity
res = activities.filter(
    target_chembl_id='CHEMBL209', activity_type='IC50',
    standard_units='nM', target_organism='Homo sapiens'
).only('molecule_chembl_id', 'canonical_smiles', 'standard_value', 'standard_units')
df_chembl = pd.DataFrame(res)
print(f"  - Fetched {len(df_chembl)} activity records from ChEMBL.")

# --- '외부 손님' 선별 및 라벨링 ---
print("\n--- Step 3: Filtering and labeling the external data... ---")
df_chembl.dropna(subset=['standard_value', 'canonical_smiles'], inplace=True)
df_chembl['standard_value'] = pd.to_numeric(df_chembl['standard_value'])
def smiles_to_inchikey(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return Chem.MolToInchiKey(mol) if mol else None
    except: return None
df_chembl['inchikey'] = df_chembl['canonical_smiles'].apply(smiles_to_inchikey)
df_chembl.dropna(subset=['inchikey'], inplace=True)

df_external = df_chembl[~df_chembl['inchikey'].isin(known_inchikeys)].copy()
df_external['label'] = df_external['standard_value'].apply(lambda x: 1 if x <= 1000 else (0 if x >= 10000 else -1))
df_external = df_external[df_external['label'] != -1]
df_external.drop_duplicates(subset=['inchikey', 'label'], inplace=True)
print(f"  - {len(df_external)} new, unique compounds selected for external validation.")
print("  - External validation set class distribution:")
print(df_external['label'].value_counts())

# --- 모델 정의 및 외부 데이터 평가 준비 ---
class GINE_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate):
        super(GINE_Model, self).__init__()
        mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINEConv(mlp1, edge_dim=num_edge_features)
        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINEConv(mlp2, edge_dim=num_edge_features)
        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels); self.bn1 = BatchNorm1d(hidden_channels); self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)
    def forward(self, x, edge_index, batch, edge_attr, global_feat):
        x = self.conv1(x, edge_index, edge_attr=edge_attr).relu(); x = self.conv2(x, edge_index, edge_attr=edge_attr).relu()
        gnn_out = global_add_pool(x, batch); combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x); x = self.fc2(x)
        return x

def get_atom_features(atom):
    return [float(f) for f in [atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(), atom.GetNumRadicalElectrons(), atom.GetHybridization(), atom.GetIsAromatic(), atom.IsInRing()]]
def get_bond_features(bond):
    return [float(f) for f in [bond.GetBondTypeAsDouble(), bond.GetIsAromatic(), bond.IsInRing(), bond.GetIsConjugated()]]
def calculate_global_descriptors(mol):
    return [Descriptors.MolLogP(mol), Descriptors.MolWt(mol), Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol), Descriptors.NumAromaticRings(mol)]
def smiles_to_graph(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return None
    atom_features = [get_atom_features(atom) for atom in mol.GetAtoms()]
    edge_indices, edge_features = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bond_feat = get_bond_features(bond)
        edge_indices.extend([[i, j], [j, i]]); edge_features.extend([bond_feat, bond_feat])
    global_features = calculate_global_descriptors(mol)
    return Data(x=torch.tensor(atom_features, dtype=torch.float), edge_index=torch.tensor(edge_indices, dtype=torch.long).t().contiguous(),
                edge_attr=torch.tensor(edge_features, dtype=torch.float), y=torch.tensor([label], dtype=torch.long),
                global_feat=torch.tensor([global_features], dtype=torch.float))

# --- 최종 저해제 모델 훈련 및 외부 데이터 평가 ---
print("\n--- Step 4: Training the final inhibitor model and evaluating... ---")
try:
    inhibitor_train_dataset = torch.load(os.path.join(BASE_DIR, "inhibitor_model_graph_dataset.pt"), weights_only=False)
    params_inhibitor = {'lr': 0.0001976113883653995, 'hidden_channels': 64, 'batch_size': 32, 'dropout_rate': 0.45631011478825545}
    final_inhibitor_model = GINE_Model(inhibitor_train_dataset[0].num_node_features, inhibitor_train_dataset[0].num_edge_features,
        inhibitor_train_dataset[0].global_feat.shape[1], params_inhibitor['hidden_channels'], params_inhibitor['dropout_rate']).to(DEVICE)
    optimizer = torch.optim.Adam(final_inhibitor_model.parameters(), lr=params_inhibitor['lr'])
    criterion = torch.nn.CrossEntropyLoss()
    train_loader = DataLoader(inhibitor_train_dataset, batch_size=params_inhibitor['batch_size'], shuffle=True)
    print("  - Training final inhibitor model...")
    for epoch in range(100):
        final_inhibitor_model.train()
        for data in train_loader:
            data = data.to(DEVICE); optimizer.zero_grad()
            out = final_inhibitor_model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
            loss = criterion(out, data.y); loss.backward(); optimizer.step()
    print("  - Model training complete.")

    print("  - Predicting on the external ChEMBL dataset...")
    external_graphs = [graph for _, row in df_external.iterrows() if (graph := smiles_to_graph(row['canonical_smiles'], row['label'])) is not None]

    if external_graphs:
        external_loader = DataLoader(external_graphs, batch_size=32)
        final_inhibitor_model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for data in external_loader:
                data = data.to(DEVICE)
                out = final_inhibitor_model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                all_labels.extend(data.y.cpu().tolist()); all_probs.extend(F.softmax(out, dim=1).cpu()[:, 1].tolist())
        external_auc = roc_auc_score(all_labels, all_probs)
        print("\n" + "="*50); print("      🎉 External Validation Result for Inhibitor Model 🎉"); print("="*50)
        print(f"  - Number of external test compounds: {len(all_labels)}")
        print(f"  - AUC on external ChEMBL data (IC50): {external_auc:.4f}")
        print("="*50)
    else:
        print("\nNo valid compounds in the external set to evaluate.")
except FileNotFoundError:
    print("Error: Could not find the inhibitor graph dataset file. Please ensure previous steps were completed.")

In [ ]:
# --- 조사 1: 라벨 일관성 확인 코드 ---
import pandas as pd
from google.colab import drive
import numpy as np

# 기본 설정 및 데이터 로딩
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"

# 우리의 저해제 모델 데이터셋 로딩
df_our_model = pd.read_csv(os.path.join(BASE_DIR, "pgp_dataset_for_inhibitor_model_v2.csv"))

# 이전에 실행했던 외부 검증 코드에서 생성된 df_external 변수가 필요합니다.
# 만약 세션이 초기화되었다면, 이전 외부 검증 코드의 1~5단계까지 다시 실행하여 df_external을 만들어주세요.
if 'df_external' in locals():
    # 비교를 위해 컬럼 이름 통일
    df_our_model.rename(columns={'desalted_inchikey': 'inchikey', 'label': 'our_label'}, inplace=True)
    df_external.rename(columns={'label': 'chembl_label'}, inplace=True)

    # InChIKey를 기준으로 두 데이터프레임 병합
    df_merged = pd.merge(df_our_model, df_external, on='inchikey', how='inner')

    # 라벨이 일치하지 않는 경우 찾기
    mismatches = df_merged[df_merged['our_label'] != df_merged['chembl_label']]

    print(f"--- 라벨 일관성 조사 결과 ---")
    print(f"우리 데이터와 ChEMBL에 공통으로 존재하는 화합물 수: {len(df_merged)}개")
    print(f"그 중 라벨이 일치하지 않는 화합물 수: {len(mismatches)}개")

    if not mismatches.empty:
        print("\n라벨 불일치 예시:")
        # 우리 모델은 1(저해제)로, ChEMBL은 0(비저해제)으로 판단한 경우
        mismatch_1_0 = mismatches[(mismatches['our_label'] == 1) & (mismatches['chembl_label'] == 0)]
        if not mismatch_1_0.empty:
            print("\n[문헌: 저해제(1) vs ChEMBL: 비저해제(0)]")
            print(mismatch_1_0[['desalted_pref_name', 'our_label', 'chembl_label', 'standard_value']].head())

        # 우리 모델은 0(비저해제)로, ChEMBL은 1(저해제)로 판단한 경우
        mismatch_0_1 = mismatches[(mismatches['our_label'] == 0) & (mismatches['chembl_label'] == 1)]
        if not mismatch_0_1.empty:
            print("\n[문헌: 비저해제(0) vs ChEMBL: 저해제(1)]")
            print(mismatch_0_1[['desalted_pref_name', 'our_label', 'chembl_label', 'standard_value']].head())
else:
    print("오류: 'df_external' 변수를 찾을 수 없습니다. 이전 외부 검증 코드 셀을 먼저 실행해주세요.")

In [ ]:
# --- 조사 2: 화학 공간 분석 코드 ---
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import drive
from rdkit import Chem
from rdkit.Chem import Descriptors
from tqdm.notebook import tqdm
import requests
import time

# --- 기본 설정 및 데이터 로딩 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"

# ✨✨✨ 빠져있던 핵심 함수 정의 부분 ✨✨✨
def get_smiles_from_inchikey(inchikey):
    """InChIKey로 PubChem에서 SMILES를 가져오는 함수"""
    if not isinstance(inchikey, str) or len(inchikey) < 10:
        return None
    try:
        # PubChem API를 통해 Isomeric SMILES를 요청합니다.
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchikey}/property/IsomericSMILES/TXT"
        response = requests.get(url, timeout=10)
        time.sleep(0.1) # 서버 부하를 줄이기 위한 지연
        if response.status_code == 200:
            return response.text.strip()
        else:
            return None
    except requests.exceptions.RequestException:
        return None

# 'df_our_model'과 'df_external' 변수가 이전 '조사 1' 셀에서 생성되었다고 가정합니다.
# 만약 런타임을 재시작했다면, '조사 1' 코드를 먼저 실행하여 두 변수를 만들어주세요.
if 'df_our_model' in locals() and 'df_external' in locals():

    def add_descriptors(df, smiles_col_name):
        """데이터프레임에 분자 특성을 계산하여 추가하는 함수"""
        # SMILES가 없는 행 제거
        df.dropna(subset=[smiles_col_name], inplace=True)
        # RDKit Mol 객체 생성 (SMILES 파싱 실패 시 None 반환)
        df['mol'] = df[smiles_col_name].apply(lambda x: Chem.MolFromSmiles(x) if pd.notna(x) else None)
        df.dropna(subset=['mol'], inplace=True) # Mol 객체 생성 실패한 행 제거

        # 분자 특성 계산
        df['MolWt'] = df['mol'].apply(Descriptors.MolWt)
        df['LogP'] = df['mol'].apply(Descriptors.MolLogP)
        df['TPSA'] = df['mol'].apply(Descriptors.TPSA)
        return df

    # 1. 외부 데이터(ChEMBL)에 분자 특성 계산
    print("ChEMBL 데이터의 분자 특성을 계산합니다...")
    df_external_desc = add_descriptors(df_external.copy(), 'canonical_smiles')

    # 2. 우리 데이터에 SMILES 정보 및 분자 특성 계산
    print("\n우리 데이터셋의 SMILES 정보를 가져오고 분자 특성을 계산합니다 (시간이 걸릴 수 있습니다)...")
    smiles_list = []
    # tqdm을 사용하여 진행 상황 표시
    for inchikey in tqdm(df_our_model['inchikey']):
        smiles = get_smiles_from_inchikey(inchikey)
        smiles_list.append(smiles)

    df_our_model['smiles'] = smiles_list
    df_our_model_desc = add_descriptors(df_our_model.copy(), 'smiles')

    # 3. 분포 시각화
    print("\n두 데이터셋의 화학 공간 분포를 시각화합니다...")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Chemical Space Distribution Comparison', fontsize=16)

    sns.kdeplot(df_our_model_desc['MolWt'], ax=axes[0], label='Our Data', fill=True, color='skyblue')
    sns.kdeplot(df_external_desc['MolWt'], ax=axes[0], label='ChEMBL Data', fill=True, color='salmon')
    axes[0].set_title('Molecular Weight Distribution')
    axes[0].legend()

    sns.kdeplot(df_our_model_desc['LogP'], ax=axes[1], label='Our Data', fill=True, color='skyblue')
    sns.kdeplot(df_external_desc['LogP'], ax=axes[1], label='ChEMBL Data', fill=True, color='salmon')
    axes[1].set_title('LogP Distribution')
    axes[1].legend()

    sns.kdeplot(df_our_model_desc['TPSA'], ax=axes[2], label='Our Data', fill=True, color='skyblue')
    sns.kdeplot(df_external_desc['TPSA'], ax=axes[2], label='ChEMBL Data', fill=True, color='salmon')
    axes[2].set_title('TPSA Distribution')
    axes[2].legend()

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

else:
    print("오류: 'df_our_model' 또는 'df_external' 변수를 찾을 수 없습니다. '조사 1' 코드 셀을 먼저 실행해주세요.")

In [ ]:

# 필요한 라이브러리 설치
!pip install chembl_webresource_client pandas rdkit-pypi torch torch_geometric scikit-learn -q


# --- 1. 라이브러리 임포트 ---
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import Descriptors
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.nn import GINEConv, global_add_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import roc_auc_score
import os
from google.colab import drive

# --- 2. 기본 설정 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# --- 3. 우리 모델의 '지인 목록' (Known InChIKeys) 만들기 ---
print("\n--- Step 1: Creating a list of known compounds... ---")
try:
    df_inhib_model = pd.read_csv(os.path.join(BASE_DIR, "pgp_dataset_for_inhibitor_model_v2.csv"))
    df_subs_model = pd.read_csv(os.path.join(BASE_DIR, "pgp_dataset_for_substrate_model_v2.csv"))
    known_inchikeys = set(df_inhib_model['desalted_inchikey'].dropna())
    known_inchikeys.update(set(df_subs_model['desalted_inchikey'].dropna()))
    print(f"  - Total of {len(known_inchikeys)} unique InChIKeys loaded.")
except FileNotFoundError as e:
    print(f"Error: Could not find dataset files. {e}")
    exit()

# --- 4. ChEMBL에서 데이터 가져오기 ---
print("\n--- Step 2: Fetching external validation data from ChEMBL... ---")
activities = new_client.activity
res = activities.filter(
    target_chembl_id='CHEMBL209', activity_type='IC50',
    standard_units='nM', target_organism='Homo sapiens'
).only('molecule_chembl_id', 'canonical_smiles', 'standard_value', 'standard_units')
df_chembl = pd.DataFrame(res)
print(f"  - Fetched {len(df_chembl)} activity records from ChEMBL.")

# --- 5. 외부 데이터 선별 및 라벨링 (✨ A안 핵심 파트 ✨) ---
print("\n--- Step 3: Filtering and labeling the external data... ---")
df_chembl.dropna(subset=['standard_value', 'canonical_smiles'], inplace=True)
df_chembl['standard_value'] = pd.to_numeric(df_chembl['standard_value'])

def smiles_to_inchikey(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return Chem.MolToInchiKey(mol) if mol else None
    except: return None
df_chembl['inchikey'] = df_chembl['canonical_smiles'].apply(smiles_to_inchikey)
df_chembl.dropna(subset=['inchikey'], inplace=True)
df_external_raw = df_chembl[~df_chembl['inchikey'].isin(known_inchikeys)].copy()
print(f"  - Found {len(df_external_raw)} new compounds not in our dataset.")

# ✨✨✨ 5-1. 화학 공간 필터링을 위한 분자 특성 계산 ✨✨✨
print("  - Calculating molecular descriptors for filtering...")
df_external_raw['mol'] = df_external_raw['canonical_smiles'].apply(Chem.MolFromSmiles)
df_external_raw.dropna(subset=['mol'], inplace=True)
df_external_raw['MolWt'] = df_external_raw['mol'].apply(Descriptors.MolWt)
df_external_raw['LogP'] = df_external_raw['mol'].apply(Descriptors.MolLogP)
df_external_raw['TPSA'] = df_external_raw['mol'].apply(Descriptors.TPSA)

# ✨✨✨ 5-2. 정의된 화학 공간 조건으로 필터링 적용 ✨✨✨
df_filtered_space = df_external_raw[
    (df_external_raw['MolWt'] <= 1000) &
    (df_external_raw['LogP'] >= -2) &
    (df_external_raw['LogP'] <= 8) &
    (df_external_raw['TPSA'] <= 250)
].copy()
print(f"  - {len(df_filtered_space)} compounds remain after filtering by chemical space.")

# 5-3. 라벨 부여 및 최종 데이터셋 생성
df_filtered_space['label'] = df_filtered_space['standard_value'].apply(
    lambda x: 1 if x <= 1000 else (0 if x >= 10000 else -1)
)
df_external = df_filtered_space[df_filtered_space['label'] != -1]
df_external.drop_duplicates(subset=['inchikey', 'label'], inplace=True)
print(f"  - {len(df_external)} final compounds selected for FAIR external validation.")
print("  - Fair external validation set class distribution:")
print(df_external['label'].value_counts())


# --- 6. 모델 정의 및 외부 데이터 평가 준비 (이전과 동일) ---
class GINE_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate):
        super(GINE_Model, self).__init__()
        mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINEConv(mlp1, edge_dim=num_edge_features)
        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINEConv(mlp2, edge_dim=num_edge_features)
        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels); self.bn1 = BatchNorm1d(hidden_channels); self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)
    def forward(self, x, edge_index, batch, edge_attr, global_feat):
        x = self.conv1(x, edge_index, edge_attr=edge_attr).relu(); x = self.conv2(x, edge_index, edge_attr=edge_attr).relu()
        gnn_out = global_add_pool(x, batch); combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x); x = self.fc2(x)
        return x

def get_atom_features(atom): return [float(f) for f in [atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(), atom.GetNumRadicalElectrons(), atom.GetHybridization(), atom.GetIsAromatic(), atom.IsInRing()]]
def get_bond_features(bond): return [float(f) for f in [bond.GetBondTypeAsDouble(), bond.GetIsAromatic(), bond.IsInRing(), bond.GetIsConjugated()]]
def calculate_global_descriptors(mol): return [Descriptors.MolLogP(mol), Descriptors.MolWt(mol), Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol), Descriptors.NumAromaticRings(mol)]
def smiles_to_graph(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return None
    atom_features = [get_atom_features(atom) for atom in mol.GetAtoms()]
    edge_indices, edge_features = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx(); bond_feat = get_bond_features(bond)
        edge_indices.extend([[i, j], [j, i]]); edge_features.extend([bond_feat, bond_feat])
    global_features = calculate_global_descriptors(mol)
    return Data(x=torch.tensor(atom_features, dtype=torch.float), edge_index=torch.tensor(edge_indices, dtype=torch.long).t().contiguous(),
                edge_attr=torch.tensor(edge_features, dtype=torch.float), y=torch.tensor([label], dtype=torch.long),
                global_feat=torch.tensor([global_features], dtype=torch.float))

# --- 7. 최종 모델 훈련 및 외부 데이터 평가 (이전과 동일) ---
print("\n--- Step 4: Training the final inhibitor model and evaluating... ---")
try:
    inhibitor_train_dataset = torch.load(os.path.join(BASE_DIR, "inhibitor_model_graph_dataset.pt"), weights_only=False)
    params_inhibitor = {'lr': 0.0001976113883653995, 'hidden_channels': 64, 'batch_size': 32, 'dropout_rate': 0.45631011478825545}
    final_inhibitor_model = GINE_Model(inhibitor_train_dataset[0].num_node_features, inhibitor_train_dataset[0].num_edge_features,
        inhibitor_train_dataset[0].global_feat.shape[1], params_inhibitor['hidden_channels'], params_inhibitor['dropout_rate']).to(DEVICE)
    optimizer = torch.optim.Adam(final_inhibitor_model.parameters(), lr=params_inhibitor['lr'])
    criterion = torch.nn.CrossEntropyLoss()
    train_loader = DataLoader(inhibitor_train_dataset, batch_size=params_inhibitor['batch_size'], shuffle=True)
    print("  - Training final inhibitor model...")
    for epoch in range(100):
        final_inhibitor_model.train()
        for data in train_loader:
            data = data.to(DEVICE); optimizer.zero_grad()
            out = final_inhibitor_model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
            loss = criterion(out, data.y); loss.backward(); optimizer.step()
    print("  - Model training complete.")

    print("  - Predicting on the FAIR external dataset...")
    external_graphs = [graph for _, row in df_external.iterrows() if (graph := smiles_to_graph(row['canonical_smiles'], row['label'])) is not None]

    if external_graphs:
        external_loader = DataLoader(external_graphs, batch_size=32)
        final_inhibitor_model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for data in external_loader:
                data = data.to(DEVICE)
                out = final_inhibitor_model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                all_labels.extend(data.y.cpu().tolist()); all_probs.extend(F.softmax(out, dim=1).cpu()[:, 1].tolist())
        external_auc = roc_auc_score(all_labels, all_probs)
        print("\n" + "="*55); print("      🎉 FAIR External Validation Result for Inhibitor Model 🎉"); print("="*55)
        print(f"  - Number of FAIR external test compounds: {len(all_labels)}")
        print(f"  - AUC on FAIR external ChEMBL data (IC50): {external_auc:.4f}")
        print("="*55)
    else:
        print("\nNo valid compounds in the filtered external set to evaluate.")
except FileNotFoundError:
    print("Error: Could not find the inhibitor graph dataset file.")

In [ ]:
# --- 1. 라이브러리 임포트 ---
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.nn import GINEConv, global_add_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split # ✨ train_test_split 추가
import os
from google.colab import drive

# --- 2. 기본 설정 ---
# 이전 셀들에서 df_external 변수가 생성되었다고 가정합니다.
# 만약 세션이 초기화되었다면, 이전 'A안' 코드의 1~5단계를 다시 실행하여 df_external을 만들어주세요.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

if 'df_external' not in locals() or df_external.empty:
    print("오류: 'df_external' 데이터가 없습니다. 이전 'A안' 코드를 먼저 실행해주세요.")
else:
    # --- 3. 통제 실험을 위한 데이터 준비 ---
    print("\n--- Control Experiment: Preparing ChEMBL-only dataset... ---")

    # 그래프 변환 함수 (이전과 동일)
    def get_atom_features(atom): return [float(f) for f in [atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(), atom.GetNumRadicalElectrons(), atom.GetHybridization(), atom.GetIsAromatic(), atom.IsInRing()]]
    def get_bond_features(bond): return [float(f) for f in [bond.GetBondTypeAsDouble(), bond.GetIsAromatic(), bond.IsInRing(), bond.GetIsConjugated()]]
    def calculate_global_descriptors(mol): return [Descriptors.MolLogP(mol), Descriptors.MolWt(mol), Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol), Descriptors.NumAromaticRings(mol)]
    def smiles_to_graph(smiles, label):
        mol = Chem.MolFromSmiles(smiles)
        if not mol: return None
        atom_features = [get_atom_features(atom) for atom in mol.GetAtoms()]
        edge_indices, edge_features = [], []
        for bond in mol.GetBonds():
            i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx(); bond_feat = get_bond_features(bond)
            edge_indices.extend([[i, j], [j, i]]); edge_features.extend([bond_feat, bond_feat])
        global_features = calculate_global_descriptors(mol)
        return Data(x=torch.tensor(atom_features, dtype=torch.float), edge_index=torch.tensor(edge_indices, dtype=torch.long).t().contiguous(),
                    edge_attr=torch.tensor(edge_features, dtype=torch.float), y=torch.tensor([label], dtype=torch.long),
                    global_feat=torch.tensor([global_features], dtype=torch.float))

    # ChEMBL 데이터로 그래프 데이터셋 생성
    chembl_graphs = [graph for _, row in df_external.iterrows() if (graph := smiles_to_graph(row['canonical_smiles'], row['label'])) is not None]
    print(f"  - Created {len(chembl_graphs)} graphs from ChEMBL data.")

    # ✨ ChEMBL 데이터를 훈련(80%) / 테스트(20%) 세트로 분할
    train_graphs, test_graphs = train_test_split(chembl_graphs, test_size=0.2, random_state=42, stratify=[g.y.item() for g in chembl_graphs])
    print(f"  - Split into {len(train_graphs)} training and {len(test_graphs)} testing graphs.")

    # --- 4. ChEMBL 데이터로 모델 훈련 및 평가 ---
    # GINE 모델 정의 (이전과 동일)
    class GINE_Model(torch.nn.Module):
        def __init__(self, num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate):
            super(GINE_Model, self).__init__()
            mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
            self.conv1 = GINEConv(mlp1, edge_dim=num_edge_features); mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
            self.conv2 = GINEConv(mlp2, edge_dim=num_edge_features); self.combined_dim = hidden_channels + num_global_features
            self.fc1 = Linear(self.combined_dim, hidden_channels); self.bn1 = BatchNorm1d(hidden_channels); self.fc2 = Linear(hidden_channels, 2)
            self.dropout = Dropout(p=dropout_rate)
        def forward(self, x, edge_index, batch, edge_attr, global_feat):
            x = self.conv1(x, edge_index, edge_attr=edge_attr).relu(); x = self.conv2(x, edge_index, edge_attr=edge_attr).relu()
            gnn_out = global_add_pool(x, batch); combined = torch.cat([gnn_out, global_feat], dim=1)
            x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x); x = self.fc2(x)
            return x

    # 모델 인스턴스 생성
    chembl_model = GINE_Model(
        chembl_graphs[0].num_node_features, chembl_graphs[0].num_edge_features,
        chembl_graphs[0].global_feat.shape[1], hidden_channels=64, dropout_rate=0.5
    ).to(DEVICE)

    optimizer = torch.optim.Adam(chembl_model.parameters(), lr=0.001)
    criterion = torch.nn.CrossEntropyLoss()
    train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_graphs, batch_size=32)

    print("\n  - Training a new model ONLY on ChEMBL data...")
    for epoch in range(100):
        chembl_model.train()
        for data in train_loader:
            data = data.to(DEVICE); optimizer.zero_grad()
            out = chembl_model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
            loss = criterion(out, data.y); loss.backward(); optimizer.step()
    print("  - Training complete.")

    # ChEMBL 테스트셋으로 평가
    print("  - Evaluating the model on the ChEMBL test set...")
    chembl_model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for data in test_loader:
            data = data.to(DEVICE)
            out = chembl_model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
            all_labels.extend(data.y.cpu().tolist()); all_probs.extend(F.softmax(out, dim=1).cpu()[:, 1].tolist())

    control_auc = roc_auc_score(all_labels, all_probs)
    print("\n" + "="*50); print("      🔬 Control Experiment Result 🔬"); print("="*50)
    print("  Model trained and tested ONLY on ChEMBL data.")
    print(f"  - AUC on ChEMBL test set: {control_auc:.4f}")
    print("="*50)

In [ ]:
# --- 1. 라이브러리 임포트 ---
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.nn import GINEConv, global_add_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.explain import Explainer, GNNExplainer
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display
from sklearn.model_selection import StratifiedKFold
import os
import requests
import time
from google.colab import drive

# --- 2. 기본 설정 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# 데이터 및 결과 경로 정의
INHIBITOR_DATASET_PATH = os.path.join(BASE_DIR, "inhibitor_model_graph_dataset.pt")
SUBSTRATE_DATASET_PATH = os.path.join(BASE_DIR, "substrate_model_graph_dataset.pt")
INHIBITOR_CSV_PATH = os.path.join(BASE_DIR, "pgp_dataset_for_inhibitor_model_v2.csv")
SUBSTRATE_CSV_PATH = os.path.join(BASE_DIR, "pgp_dataset_for_substrate_model_v2.csv")
XAI_OUTPUT_DIR = os.path.join(BASE_DIR, "XAI_Analysis_Results")
os.makedirs(XAI_OUTPUT_DIR, exist_ok=True)

# 최적 하이퍼파라미터
params_inhibitor = {'lr': 0.0001976113883653995, 'hidden_channels': 64, 'batch_size': 32, 'dropout_rate': 0.45631011478825545}
params_substrate = {'lr': 0.00024644109016033717, 'hidden_channels': 64, 'batch_size': 16, 'dropout_rate': 0.5269097585930824}

# --- 3. GINE 모델 클래스 정의 ---
class GINE_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate):
        super(GINE_Model, self).__init__()
        mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINEConv(mlp1, edge_dim=num_edge_features)
        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINEConv(mlp2, edge_dim=num_edge_features)
        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels); self.bn1 = BatchNorm1d(hidden_channels); self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)
    def forward(self, x, edge_index, batch, edge_attr, global_feat):
        x = self.conv1(x, edge_index, edge_attr=edge_attr).relu(); x = self.conv2(x, edge_index, edge_attr=edge_attr).relu()
        gnn_out = global_add_pool(x, batch); combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x); x = self.fc2(x)
        return x

# --- 4. OOF 예측 및 분석 대상 선정 ---
def get_oof_predictions(dataset, csv_path, params, model_name):
    print(f"\nRunning Out-of-Fold predictions for {model_name}...")
    df_names = pd.read_csv(csv_path)
    names_list = df_names['desalted_pref_name'].tolist()[:len(dataset)]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    oof_preds = np.zeros(len(dataset))
    for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(len(dataset)), labels), 1):
        train_subset = [dataset[i] for i in train_idx]; val_subset = [dataset[i] for i in val_idx]
        model = GINE_Model(dataset[0].num_node_features, dataset[0].num_edge_features, dataset[0].global_feat.shape[1],
                         params['hidden_channels'], params['dropout_rate']).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'])
        criterion = torch.nn.CrossEntropyLoss()
        train_loader = DataLoader(train_subset, batch_size=params['batch_size'], shuffle=True, drop_last=True)
        for epoch in range(100):
            model.train()
            for data in train_loader:
                data = data.to(DEVICE); optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                loss = criterion(out, data.y); loss.backward(); optimizer.step()
        model.eval()
        temp_probs = []
        val_loader = DataLoader(val_subset, batch_size=params['batch_size'])
        with torch.no_grad():
            for data in val_loader:
                data = data.to(DEVICE)
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                probs = F.softmax(out, dim=1)[:, 1].cpu().numpy()
                temp_probs.extend(probs)
        oof_preds[val_idx] = temp_probs
    return pd.DataFrame({'name': names_list, 'prob': oof_preds})

inhibitor_dataset = torch.load(INHIBITOR_DATASET_PATH, weights_only=False)
substrate_dataset = torch.load(SUBSTRATE_DATASET_PATH, weights_only=False)
inhibitor_preds_df = get_oof_predictions(inhibitor_dataset, INHIBITOR_CSV_PATH, params_inhibitor, "Model-1 (Inhibitor)").rename(columns={'prob': 'P_inhibition'})
substrate_preds_df = get_oof_predictions(substrate_dataset, SUBSTRATE_CSV_PATH, params_substrate, "Model-2 (Substrate)").rename(columns={'prob': 'P_substrate'})
final_results = pd.merge(inhibitor_preds_df, substrate_preds_df, on='name', how='inner')

inhibitors = final_results[(final_results['P_inhibition'] >= 0.7) & (final_results['P_substrate'] <= 0.3)]
substrates = final_results[(final_results['P_inhibition'] <= 0.3) & (final_results['P_substrate'] >= 0.7)]
inhibitor_targets = inhibitors.sort_values('P_inhibition', ascending=False).head(15)['name'].tolist()
substrate_targets = substrates.sort_values('P_substrate', ascending=False).head(15)['name'].tolist()
print("\n--- SAR 분석 대상 선정 완료 ---")
print(f"🕵️‍♂️ 특이적 저해제 (상위 {len(inhibitor_targets)}개): {inhibitor_targets}")
print(f" সাক্ষী 특이적 기질 (상위 {len(substrate_targets)}개): {substrate_targets}")

# --- 5. XAI 분석을 위한 최종 모델 훈련 ---
def train_final_model(dataset, params):
    model = GINE_Model(dataset[0].num_node_features, dataset[0].num_edge_features, dataset[0].global_feat.shape[1],
                       params['hidden_channels'], params['dropout_rate']).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'])
    criterion = torch.nn.CrossEntropyLoss()
    loader = DataLoader(dataset, batch_size=params['batch_size'], shuffle=True)
    for epoch in range(100):
        model.train()
        for data in loader:
            data = data.to(DEVICE); optimizer.zero_grad()
            out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
            loss = criterion(out, data.y); loss.backward(); optimizer.step()
    return model

print("\n--- XAI 분석용 최종 모델 훈련 ---")
final_inhibitor_model = train_final_model(inhibitor_dataset, params_inhibitor)
final_substrate_model = train_final_model(substrate_dataset, params_substrate)

# --- ✨✨✨ 6. XAI 분석 사전 준비: SMILES 정보 추가 ✨✨✨ ---
def get_smiles_from_inchikey(inchikey):
    if not isinstance(inchikey, str) or len(inchikey) < 10: return None
    try:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchikey}/property/IsomericSMILES/TXT"
        response = requests.get(url, timeout=10)
        time.sleep(0.05)
        return response.text.strip() if response.status_code == 200 else None
    except requests.exceptions.RequestException: return None

def add_smiles_to_df(df):
    print(f"  - {len(df)}개 화합물에 대한 SMILES 정보 가져오는 중...")
    smiles_list = df['desalted_inchikey'].apply(get_smiles_from_inchikey)
    df['smiles'] = smiles_list
    df.dropna(subset=['smiles'], inplace=True)
    return df

print("\n--- XAI 분석용 데이터프레임에 SMILES 정보 추가 ---")
df_inhibitor_full = pd.read_csv(INHIBITOR_CSV_PATH)
df_inhibitor_full = add_smiles_to_df(df_inhibitor_full)

df_substrate_full = pd.read_csv(SUBSTRATE_CSV_PATH)
df_substrate_full = add_smiles_to_df(df_substrate_full)

# 그래프 객체에도 이름과 SMILES 정보 부여
for df_full, dataset_graphs in [(df_inhibitor_full, inhibitor_dataset), (df_substrate_full, substrate_dataset)]:
    # Create a mapping from inchikey to name and smiles for faster lookup
    name_map = df_full.set_index('desalted_inchikey')['desalted_pref_name'].to_dict()
    smiles_map = df_full.set_index('desalted_inchikey')['smiles'].to_dict()

    for g in dataset_graphs:
        # Find the corresponding inchikey in the full dataset for this graph object
        # This assumes the order of graphs in .pt file is the same as rows in the .csv file
        # which is a strong but necessary assumption here.
        idx = inhibitor_dataset.index(g) if g in inhibitor_dataset else substrate_dataset.index(g)
        inchikey = df_full.iloc[idx]['desalted_inchikey']

        g.name = name_map.get(inchikey)
        g.smiles = smiles_map.get(inchikey)

# --- 7. XAI 분석 및 이미지 저장 자동화 ---
def run_xai_and_save(target_names, group_name, model, full_dataset):
    group_output_dir = os.path.join(XAI_OUTPUT_DIR, group_name)
    os.makedirs(group_output_dir, exist_ok=True)
    print(f"\nAnalyzing group: '{group_name}' (saving to {group_output_dir})")

    explainer = Explainer(model=model, algorithm=GNNExplainer(epochs=200),
                          explanation_type='model', node_mask_type='attributes',
                          model_config=dict(mode='binary_classification', task_level='graph', return_type='raw'))

    for compound_name in target_names:
        try:
            target_graph = next(g for g in full_dataset if hasattr(g, 'name') and g.name == compound_name)

            explanation = explainer(x=target_graph.x.to(DEVICE), edge_index=target_graph.edge_index.to(DEVICE),
                                    batch=torch.zeros(target_graph.x.size(0), dtype=torch.long).to(DEVICE),
                                    edge_attr=target_graph.edge_attr.to(DEVICE), global_feat=target_graph.global_feat.to(DEVICE))

            mol = Chem.MolFromSmiles(target_graph.smiles)
            node_mask = explanation.node_mask.cpu().numpy().flatten()
            norm_mask = (node_mask - node_mask.min()) / (node_mask.max() - node_mask.min() + 1e-9)
            atom_colors = {i: (1.0, 1.0 - v, 0.0) for i, v in enumerate(norm_mask)}

            img = Draw.MolToImage(mol, highlightAtoms=list(range(mol.GetNumAtoms())), highlightAtomColors=atom_colors, size=(400, 400))

            safe_filename = "".join([c for c in compound_name if c.isalnum() or c in (' ', '-')]).rstrip()
            output_path = os.path.join(group_output_dir, f"{safe_filename}.png")
            img.save(output_path)
            print(f"  - Saved analysis for: {compound_name}")

        except StopIteration: print(f"  - Skipping {compound_name}: Not found in graph dataset.")
        except Exception as e: print(f"  - An error occurred for {compound_name}: {e}")

run_xai_and_save(inhibitor_targets, "Specific_Inhibitors", final_inhibitor_model, inhibitor_dataset)
run_xai_and_save(substrate_targets, "Specific_Substrates", final_substrate_model, substrate_dataset)

print("\n\n--- ✅ 모든 분석 및 저장이 완료되었습니다! ---")
print(f"결과는 Google Drive의 다음 폴더에서 확인하실 수 있습니다:")
print(XAI_OUTPUT_DIR)